# Comparaison Random Forest vs régression linéaire — prédiction de `Progress`

Ce notebook **évalue et compare** deux modèles entraînés sur les **mêmes données** et le **même découpage train/test** (`test_size=0.2`, `random_state=42`) que les scripts `Random Forest/MODEL/rf_progress_pipeline.py` et `Régression Linéaire/MODEL/lr_progress_pipeline.py`.

**Contenu :** métriques (R², RMSE, MAE, MAPE/SMAPE), tableaux comparatifs, graphiques (nuages de points, résidus, boxplot, erreur cumulative, importances RF) et **interprétation** en français.

## 1. Données et prétraitement

Nettoyage, dates, encodage ordinal / one-hot, IQR — **aligné** sur les pipelines du projet. Les prédictions sur le jeu de test proviennent des modèles sauvegardés (`.pkl`) pour garantir la cohérence avec l’entraînement déjà effectué.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

# Racine : dossier "Prediction avancement"
NB_ROOT = Path.cwd().resolve()
if not (NB_ROOT / "shared" / "progress_inference.py").is_file():
    NB_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else NB_ROOT

import sys

sys.path.insert(0, str(NB_ROOT / "shared"))
from progress_inference import ml_project_root  # noqa: E402

ML_ROOT = ml_project_root(NB_ROOT / "Comparaison_RF_LR_Progress.ipynb")
DATA_PATH = ML_ROOT / "Project-Management-2-enriched.csv"
RF_MODEL_DIR = NB_ROOT / "Random Forest" / "MODEL"
LR_MODEL_DIR = NB_ROOT / "Régression Linéaire" / "MODEL"
FIG_DIR = NB_ROOT / "presentation_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Progress"
REFERENCE_DATE = pd.Timestamp("2026-04-12")
ORDER_PROJECT_STATUS = ["On Hold", "Behind", "On Track", "Completed"]
ORDER_TASK_STATUS = ["Pending", "In Progress", "Completed"]
COLS_IQR = ["Hours Spent", "Budget", "Actual Cost"]
DROP_LEAKAGE_AND_IDS = [
    "Risk_Level",
    "Project ID",
    "Project Name",
    "Task Name",
    "Location",
]


def cap_iqr(series: pd.Series, k: float = 1.5) -> pd.Series:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0 or np.isnan(iqr):
        return series
    low, high = q1 - k * iqr, q3 + k * iqr
    return series.clip(lower=low, upper=high)


def preprocess_progress_df(csv_path: Path) -> tuple[pd.DataFrame, list[str]]:
    """Même logique que rf_progress_pipeline.py / lr_progress_pipeline.py."""
    df = pd.read_csv(csv_path)
    df = df.drop_duplicates()

    n = len(df)
    rows_with_na = df.isnull().any(axis=1).sum()
    pct_na_rows = 100.0 * rows_with_na / n if n else 0.0
    if rows_with_na > 0:
        if pct_na_rows < 5.0:
            df = df.dropna()
        else:
            for c in df.select_dtypes(include=[np.number]).columns:
                df[c] = df[c].fillna(df[c].median())
            for c in df.select_dtypes(include=["object"]).columns:
                mode = df[c].mode()
                df[c] = df[c].fillna(mode.iloc[0] if len(mode) else "")

    const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    if const_cols:
        df = df.drop(columns=const_cols)

    for col in COLS_IQR:
        if col in df.columns:
            df[col] = cap_iqr(df[col])

    for c in ["Start Date", "End Date"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], dayfirst=True, errors="coerce")

    if "Start Date" in df.columns and "End Date" in df.columns:
        df["planned_duration_days"] = (df["End Date"] - df["Start Date"]).dt.days
        df["days_since_start"] = (REFERENCE_DATE - df["Start Date"]).dt.days
        df["remaining_days"] = (df["End Date"] - REFERENCE_DATE).dt.days
        df = df.drop(columns=["Start Date", "End Date"])
        if "Planned_Duration_Days" in df.columns:
            df = df.drop(columns=["Planned_Duration_Days"])

    to_drop = [c for c in DROP_LEAKAGE_AND_IDS if c in df.columns]
    df = df.drop(columns=to_drop, errors="ignore")

    high_card = []
    for c in df.select_dtypes(include=["object"]).columns:
        if c == TARGET:
            continue
        if df[c].nunique(dropna=False) > 10:
            high_card.append(c)
    if high_card:
        df = df.drop(columns=high_card, errors="ignore")

    if "Project Status" in df.columns:
        ps_map = {v: i for i, v in enumerate(ORDER_PROJECT_STATUS)}
        unk = len(ORDER_PROJECT_STATUS)
        df["Project_Status_ord"] = df["Project Status"].map(ps_map).fillna(unk).astype(int)
        df = df.drop(columns=["Project Status"])

    if "Task Status" in df.columns:
        ts_map = {v: i for i, v in enumerate(ORDER_TASK_STATUS)}
        unk = len(ORDER_TASK_STATUS)
        df["Task_Status_ord"] = df["Task Status"].map(ts_map).fillna(unk).astype(int)
        df = df.drop(columns=["Task Status"])

    cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if cat_cols:
        df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

    if TARGET not in df.columns:
        raise ValueError(f"Colonne cible '{TARGET}' absente.")

    y_raw = df[TARGET].astype(float)
    if y_raw.min() < 0 or y_raw.max() > 1:
        ymin, ymax = y_raw.min(), y_raw.max()
        df[TARGET] = (y_raw - ymin) / (ymax - ymin) if ymax > ymin else y_raw.clip(0, 1)
    else:
        df[TARGET] = y_raw.clip(0, 1)

    feature_names = [c for c in df.columns if c != TARGET]
    return df, feature_names


df_full, _ = preprocess_progress_df(DATA_PATH)
X = df_full.drop(columns=[TARGET])
y = df_full[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_path = RF_MODEL_DIR / "rf_progress_model.pkl"
lr_path = LR_MODEL_DIR / "lr_progress_model.pkl"
rf_feat_path = RF_MODEL_DIR / "rf_progress_features.pkl"

if not rf_path.is_file() or not lr_path.is_file():
    raise FileNotFoundError(
        "Modèles .pkl introuvables. Exécuter d’abord rf_progress_pipeline.py puis lr_progress_pipeline.py."
    )

model_rf = joblib.load(rf_path)
pipe_lr = joblib.load(lr_path)
expected_rf_cols = list(joblib.load(rf_feat_path))

missing_rf = [c for c in expected_rf_cols if c not in X_test.columns]
if missing_rf:
    raise ValueError(f"Colonnes manquantes pour le RF : {missing_rf}")

X_test_rf = X_test[expected_rf_cols]
y_test_np = np.asarray(y_test, dtype=float)
y_pred_rf = model_rf.predict(X_test_rf)
y_pred_lr = pipe_lr.predict(X_test)

print("Taille test :", len(y_test_np))
print("Progress test — min, max, moyenne :", y_test_np.min(), y_test_np.max(), y_test_np.mean())
print("Nombre de y_test == 0 :", int(np.sum(y_test_np == 0)))

## 2. Métriques (sklearn)

- **R²**, **RMSE** (`mean_squared_error` + racine), **MAE**.
- **MAPE** : si des valeurs réelles sont **0**, la MAPE classique diverge ; on affiche alors **SMAPE** (symétrique, stable) et une **MAPE restreinte** aux observations avec `|y| > ε`.

In [ ]:
EPS = 1e-8


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def mape_masked(y_true, y_pred, eps: float = EPS) -> float | None:
    mask = np.abs(y_true) > eps
    if not np.any(mask):
        return None
    return float(
        np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0
    )


def smape(y_true, y_pred, eps: float = EPS) -> float:
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(200.0 * np.abs(y_true - y_pred) / denom))


def metrics_row(name: str, y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    m_mape = mape_masked(y_true, y_pred)
    # sklearn MAPE (peut exploser si y=0)
    try:
        sk_mape = float(mean_absolute_percentage_error(y_true, y_pred)) * 100.0
        if not np.isfinite(sk_mape) or sk_mape > 1e6:
            sk_mape = np.nan
    except Exception:
        sk_mape = np.nan
    return {
        "Modèle": name,
        "R²": r2_score(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE (% |y|>ε)": m_mape,
        "SMAPE (%)": smape(y_true, y_pred),
        "MAPE sklearn (%)": sk_mape,
    }


rows = [
    metrics_row("Régression linéaire", y_test_np, y_pred_lr),
    metrics_row("Random Forest", y_test_np, y_pred_rf),
]
df_metrics = pd.DataFrame(rows)
df_display = df_metrics[
    ["Modèle", "R²", "RMSE", "MAE", "MAPE (% |y|>ε)", "SMAPE (%)"]
].copy()
df_display

### Tableau demandé (R², RMSE, MAE ; MAPE = version robuste)

Colonne **MAPE (%)** : moyenne de `|y - ŷ| / |y| × 100` sur les lignes où `|y| > ε`. Si toutes les valeurs de `y` sont nulles sur le test, utiliser **SMAPE**.

In [ ]:
summary = pd.DataFrame(
    {
        "Modèle": ["Régression linéaire", "Random Forest"],
        "R²": [rows[0]["R²"], rows[1]["R²"]],
        "RMSE": [rows[0]["RMSE"], rows[1]["RMSE"]],
        "MAE": [rows[0]["MAE"], rows[1]["MAE"]],
        "MAPE (%)": [
            rows[0]["MAPE (% |y|>ε)"] if rows[0]["MAPE (% |y|>ε)"] is not None else rows[0]["SMAPE (%)"],
            rows[1]["MAPE (% |y|>ε)"] if rows[1]["MAPE (% |y|>ε)"] is not None else rows[1]["SMAPE (%)"],
        ],
    }
)
summary["R²"] = summary["R²"].map(lambda x: round(float(x), 4))
summary["RMSE"] = summary["RMSE"].map(lambda x: round(float(x), 4))
summary["MAE"] = summary["MAE"].map(lambda x: round(float(x), 4))
summary["MAPE (%)"] = summary["MAPE (%)"].map(
    lambda x: round(float(x), 2) if x is not None and np.isfinite(x) else np.nan
)
summary

## 3. Graphiques

### a) Réel vs prédit (diagonale `y = x`)

In [ ]:
def plot_actual_vs_pred(y_true, y_pred, title: str, fname: str):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true, y_pred, alpha=0.65, edgecolors="k", linewidths=0.3, s=40)
    lo = float(min(y_true.min(), y_pred.min()))
    hi = float(max(y_true.max(), y_pred.max()))
    ax.plot([lo, hi], [lo, hi], "k--", lw=1.5, label="y = x")
    ax.set_xlabel("Vraies valeurs (y_test)")
    ax.set_ylabel("Valeurs prédites")
    ax.set_title(title)
    ax.legend()
    ax.set_aspect("equal", adjustable="box")
    plt.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()


plot_actual_vs_pred(
    y_test_np, y_pred_lr, "Régression linéaire — réel vs prédit", "lr_actual_vs_pred.png"
)
plot_actual_vs_pred(
    y_test_np, y_pred_rf, "Random Forest — réel vs prédit", "rf_actual_vs_pred.png"
)

### b) Histogramme des résidus + KDE

In [ ]:
def plot_residual_hist(y_true, y_pred, title: str, fname: str):
    resid = np.asarray(y_true).ravel() - np.asarray(y_pred).ravel()
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(resid, kde=True, stat="density", ax=ax, color="steelblue", edgecolor="black")
    ax.axvline(0, color="crimson", ls="--", lw=1.2, label="0")
    ax.set_xlabel("Résidu (y_test − y_pred)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()


plot_residual_hist(y_test_np, y_pred_lr, "Résidus — régression linéaire", "lr_residuals_hist.png")
plot_residual_hist(y_test_np, y_pred_rf, "Résidus — Random Forest", "rf_residuals_hist.png")

### c) Boxplot comparatif des résidus

In [ ]:
res_lr = y_test_np - y_pred_lr
res_rf = y_test_np - y_pred_rf
df_box = pd.DataFrame(
    {
        "Résidu": np.concatenate([res_lr, res_rf]),
        "Modèle": ["Régression linéaire"] * len(res_lr) + ["Random Forest"] * len(res_rf),
    }
)
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df_box, x="Modèle", y="Résidu", ax=ax, palette=["#8ecae6", "#219ebc"])
ax.axhline(0, color="gray", ls="--", lw=1)
ax.set_title("Comparaison des résidus (jeu test)")
plt.tight_layout()
fig.savefig(FIG_DIR / "boxplot_residuals_both.png", dpi=150, bbox_inches="tight")
plt.show()

### d) Courbe d’erreur absolue cumulative (tri croissant)

In [ ]:
def cumulative_abs_error_curve(y_true, y_pred, label: str, ax):
    err = np.sort(np.abs(np.asarray(y_true).ravel() - np.asarray(y_pred).ravel()))
    cum = np.cumsum(err)
    cum_pct = 100.0 * cum / cum[-1] if cum[-1] > 0 else np.zeros_like(cum)
    pct_samples = 100.0 * (np.arange(1, len(err) + 1) / len(err))
    ax.plot(pct_samples, cum_pct, label=label, lw=2)


fig, ax = plt.subplots(figsize=(7, 4))
cumulative_abs_error_curve(y_test_np, y_pred_lr, "Régression linéaire", ax)
cumulative_abs_error_curve(y_test_np, y_pred_rf, "Random Forest", ax)
ax.set_xlabel("Pourcentage d’échantillons test (erreur absolue triée croissante)")
ax.set_ylabel("% de l’erreur absolue totale cumulée")
ax.set_title("Courbe d’erreur cumulative")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "cumulative_abs_error.png", dpi=150, bbox_inches="tight")
plt.show()

### e) Importance des variables — Random Forest (top 3)

In [ ]:
importances = model_rf.feature_importances_
names = expected_rf_cols
ranked = sorted(zip(importances, names), key=lambda t: t[0], reverse=True)
top3 = ranked[:3][::-1]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh([n for _, n in top3], [i for i, _ in top3], color="teal", edgecolor="black")
ax.set_xlabel("Importance")
ax.set_title("Random Forest — 3 variables les plus importantes")
plt.tight_layout()
fig.savefig(FIG_DIR / "rf_feature_importance_top3.png", dpi=150, bbox_inches="tight")
plt.show()
print("Top 3 :", [(n, round(float(i), 4)) for i, n in ranked[:3]])

---

## 4. Interprétation (résultats numériques ci-dessus)

Les phrases ci-dessous doivent être **complétées après exécution** du notebook (valeurs affichées dans `summary` et graphiques). Voici un **modèle d’analyse** cohérent avec ce type de problème (régression sur un taux d’avancement entre 0 et 1) :

### Quel modèle obtient le meilleur R² et le plus faible RMSE ? Pourquoi ?

En général, le **Random Forest** obtient un **R² plus élevé** et un **RMSE plus faible** que la **régression linéaire**. La régression linéaire suppose une relation approximativement **linéaire** entre les features et `Progress` ; or l’avancement dépend souvent de **seuils**, de **non-linéarités** (ex. saturation) et d’**interactions** (statuts, durées, coûts). Le RF, par ses partitions récursives et la moyenne de nombreux arbres, **approxime** ces effets sans les formuler explicitement.

### Résidus du modèle linéaire : centrés sur zéro ? Hétéroscédasticité ?

Inspecter l’**histogramme** et le **boxplot** : la **médiane** des résidus linéaires est souvent proche de 0 (la régression des moindres carrés impose une somme de résidus nulle avec intercept, mais pas la symétrie parfaite). Si le nuage **réel vs prédit** s’écarte systématiquement de la diagonale pour certaines plages de `Progress` (ex. sous-estimation des valeurs élevées), cela traduit une **tendance** (biais structurel) plutôt qu’un simple bruit aléatoire. Une **forme en entonnoir** dans un graphe résidu vs prédit (non tracé ici mais utile en prolongement) suggérerait de l’**hétéroscédasticité**.

### Le Random Forest améliore-t-il significativement les prédictions ?

Comparer **RMSE** et **MAE** dans le tableau : une baisse notable de RMSE indique moins d’**erreurs extrêmes**. Le RF capture des **non-linéarités** et des **interactions** entre variables sans les encoder à la main ; la régression linéaire, même avec `StandardScaler`, ne modélise qu’une **combinaison linéaire** des entrées.

### Variables les plus importantes (RF) et sens métier

Les **importances** (barres horizontales) mettent souvent en avant des variables liées au **calendrier** (`days_since_start`, `remaining_days`, `planned_duration_days`), au **coût / budget** ou au **temps passé** (`Hours Spent`, `Budget`, `Actual Cost`, `Budget_Utilization`) et aux **statuts** ordinals. C’est **cohérent** avec l’intuition : l’avancement dépend de **où on en est dans le planning**, des **ressources consommées** et de l’**état** du projet/tâche. Vérifier sur votre sortie si le **top 3** correspond bien à cette lecture.

### Limites de l’évaluation

- **Taille du jeu de test** (~20 % des lignes) : estimateurs de R² / RMSE **bruités** si peu d’observations.
- **Distribution de `Progress`** : si peu de valeurs proches de 0 ou de 1, la **MAPE** masque des comportements différents sur les extrêmes ; d’où l’usage conjoint de **MAE** et **SMAPE**.
- **Fuites de données** : les pipelines excluent déjà certaines colonnes ; toute variable trop proche de la cible fausserait l’évaluation.
- **Généralisation** : performances valables pour ce **CSV** et ce **prétraitement** ; d’autres populations de projets peuvent dégrader les métriques.

---

**Figures enregistrées dans :** `Prediction avancement/presentation_figures/` (PNG, 150 dpi).